### Tools


In [ ]:
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
from langchain_groq import ChatGroq

model = ChatGroq(model="qwen/qwen3-32b")
model


ChatGroq(metadata={'versions': {'langchain-core': '1.4.6', 'langchain': '1.3.7'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x110b5a660>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x110b5b0e0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [1]:
from langchain.chat_models import init_chat_model

agent = init_chat_model(model="groq:qwen/qwen3-32b")
response = agent.invoke("Richest man in the world?")
response.content

"<think>\nOkay, the user is asking about the richest person in the world. Let me start by recalling the latest information. As of my last update in October 2023, Elon Musk was the richest person globally. But I should check if there have been any significant changes since then. Bill Gates and Jeff Bezos are also in the top ranks, so I need to consider their current net worths.\n\nI remember that Elon Musk's wealth is tied to Tesla and SpaceX. If Tesla's stock has been performing well or there have been major developments in SpaceX, that could affect his net worth. On the other hand, Jeff Bezos' Amazon might have grown, or maybe there have been changes in his other ventures like Blue Origin. Similarly, Bernard Arnault from LVMH has been rising in the rankings due to luxury goods market performance.\n\nI should also consider external factors like market trends, company stock prices, and any recent acquisitions or losses. For example, if there's been a dip in the tech sector, that might a

### Tools calling

from langchain_tools import tools --> now this tool can be used as a decorator @tool and any funciton can be used a tool


In [ ]:
from langchain.tools import tool


@tool
def get_weather(city: str) -> str:
    """Get weather of the city"""
    return f"The weather of {city} is sunny!"


model_with_tools = model.bind_tools([get_weather])


In [5]:
response = model_with_tools.invoke("What is the weather like in Boston?")

response


AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking about the weather in Boston. I need to use the get_weather function. Let me check the function parameters. It requires a city name. The user mentioned Boston, so I should call the function with "Boston" as the city argument. Make sure the JSON is correctly formatted with the city parameter. I\'ll structure the tool_call accordingly.\n', 'tool_calls': [{'id': 'qgqmcnp7q', 'function': {'arguments': '{"city":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 98, 'prompt_tokens': 153, 'total_tokens': 251, 'completion_time': 0.164540567, 'completion_tokens_details': {'reasoning_tokens': 74}, 'prompt_time': 0.00607509, 'prompt_tokens_details': None, 'queue_time': 0.04897671, 'total_time': 0.170615657}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs':

### Tool execution loop


In [ ]:
# Step 1: Model Generates tool calls
messages = [{"role": "user", "content": "What is the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
print("ai_msg: ", ai_msg)
messages.append(ai_msg)
print("messages: ", messages)


# Step 2: Exectue Tools and collect results
for tool_call in ai_msg.tool_calls:
    # Excecute the tool with generated arguments
    print("tool_call:", tool_call)
    tool_result = get_weather.invoke(tool_call)
    print("tool_result: ", tool_result)
    messages.append(tool_result)
    print("messages: ", messages)

final_response = model_with_tools.invoke(messages)
print("final_response: ", final_response)

ai_msg:  content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking about the weather in Boston. Let me check the tools provided. There\'s a function called get_weather that takes a city parameter. Since the user mentioned Boston, I need to call this function with "Boston" as the city argument. I\'ll make sure to format the tool call correctly within the XML tags as specified. No other tools are available, so this should be the only function call needed.\n', 'tool_calls': [{'id': 'ep77qm6h7', 'function': {'arguments': '{"city":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 110, 'prompt_tokens': 152, 'total_tokens': 262, 'completion_time': 0.18503213, 'completion_tokens_details': {'reasoning_tokens': 86}, 'prompt_time': 0.006325776, 'prompt_tokens_details': None, 'queue_time': 0.045982964, 'total_time': 0.191357906}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 